# 4 OOP (Objekt-orientierte Programmierung)

# 4.1 Konzepte

## Klassen und Instanzen (4)

In [ ]:
class Playlist:
    def __init__(self, title):
        self.title = title
        self.tracks = []

    def add_track(self, track):
        self.tracks.append(track)

morning_mix = Playlist("Morning Mix")
morning_mix.add_track("Blue in Green")

focus_mix = Playlist("Focus Mix")
focus_mix.add_track("Autumn Leaves")
focus_mix.add_track("Solar")

print(morning_mix.title, morning_mix.tracks)
print(focus_mix.title, focus_mix.tracks)
print(type(morning_mix).__name__)

## Attribute, Konstruktoren und `self` (5)

In [ ]:
class Article:
    tax_rate = 0.19

    def __init__(self, name, net_price):
        self.name = name
        self.net_price = net_price

    def gross_price(self):
        return self.net_price * (1 + self.tax_rate)

    def __str__(self):
        return f"{self.name}: {self.net_price} EUR (net), {self.gross_price():.2f} EUR (gross)"

    @classmethod
    def increase_tax_rate(cls, value):
        cls.tax_rate += value

    @staticmethod
    def is_valid_price(value):
        return value >= 0


coffee = Article("Coffee Beans", 12.5)
tea = Article("Green Tea", 8.0)
print(coffee)
print(tea)

In [ ]:
Article.increase_tax_rate(0.01)
print(coffee)
print(tea)

In [ ]:
tea.tax_rate = 0.07
print(tea)

In [ ]:
print(coffee)

In [ ]:
print(Article.is_valid_price(-3))
print(tea.is_valid_price(42))

## Referenzen und Kopien (6)

In [ ]:
import copy

class PackingList:
    def __init__(self, items):
        self.items = items

original = PackingList(["notebook", "charger"])
alias = original
shallow_copy = copy.copy(original)
deep_copy = copy.deepcopy(original)

alias.items.append("water bottle")
shallow_copy.items.append("snacks")
deep_copy.items.append("jacket")

print("original:", original.items)
print("alias:", alias.items)
print("shallow_copy:", shallow_copy.items)
print("deep_copy:", deep_copy.items)

In [ ]:
print("alias is original:", alias is original)
print("shallow_copy is original:", shallow_copy is original)
print("shallow shares list:", shallow_copy.items is original.items)
print("deep shares list:", deep_copy.items is original.items)

## Übungen zu 4.1

### Aufgabe 1 - Klassenattribute und Objektzustand

1. Schreibe eine Klasse `Course`.
2. Jedes Objekt hat `title` und eine Liste `participants`.
3. Alle Kurse teilen das Klassenattribut `semester = "WS 2026/27"`.
4. Implementiere `enroll(name)` und `participant_count()`, sowie `__str__()`.
5. Erzeuge zwei Kurse und zeige, dass Teilnehmerlisten getrennt, `semester` aber gemeinsam ist.

# 4.2 Polymorphie

## Dynamisches Binden (9)

In [ ]:
class NotificationChannel:
    def deliver(self, message: str) -> str:
        return f"base:{message}"

class EmailChannel(NotificationChannel):
    def deliver(self, message: str) -> str:
        return f"email:{message.lower()}"

class SmsChannel(NotificationChannel):
    def deliver(self, message: str) -> str:
        return f"sms:{message[:10]}"

channels: list[NotificationChannel] = [EmailChannel(), SmsChannel()]
for channel in channels:
    print(channel.deliver("System maintenance at 18:00"))

## Vererbung und `super()` (10)

In [ ]:
class Person:
    def __init__(self, name):
        self.name = name

class Tutor(Person):
    def __init__(self, name, subject):
        super().__init__(name)
        self.subject = subject

    def introduce(self):
        return f"{self.name} teaches {self.subject}"

tutor = Tutor("Ada", "Python")
print(tutor.introduce())
print(isinstance(tutor, Person))

## Mehrfachvererbung und MRO (11)

In [ ]:
class A:
    def describe(self):
        return "A"

class B(A):
    def describe(self):
        return "B -> " + super().describe()

class C(A):
    def describe(self):
        return "C -> " + super().describe()

print(B().describe())
print(C().describe())

In [ ]:
class D(B, C):
    def describe(self):
        return "D -> " + super().describe()

print(D().describe())
print([cls.__name__ for cls in D.mro()])

## Interfaces (12)

In [ ]:
from abc import ABC, abstractmethod

class Formatter(ABC):
    @abstractmethod
    def format(self, text: str) -> str:
        raise NotImplementedError

class UpperFormatter(Formatter):
    def format(self, text: str) -> str:
        return text.upper()

formatter = UpperFormatter()
print(formatter.format("Objekte und Methoden"))

In [ ]:
try:
    Formatter()
except TypeError as error:
    print("ERROR", error)

## Overriding und Overloading (13)

In [ ]:
## Overriding

class Notification:
    def render(self) -> str:
        return "generic notification"

class EmailNotification(Notification):
    def render(self) -> str:
        return "email notification"

print(Notification().render())
print(EmailNotification().render())


In [ ]:
## "Overloading" geht nicht

class BrokenNameFormatter:
    def format_name(self, name):
        return name

    # "Destroys" the previous method, it is not an overload!
    def format_name(self, first_name, last_name):
        return f"{first_name} {last_name}"

broken = BrokenNameFormatter()
try:
    print(broken.format_name("Ada"))
except TypeError as error:
    print("ERROR", error)

In [ ]:
## Stattdessen: Flexible Parameter

class FlexibleNameFormatter:
    def format_name(self, first_name, last_name=None, *, uppercase=False):
        full_name = first_name if last_name is None else f"{first_name} {last_name}"
        return full_name.upper() if uppercase else full_name

formatter = FlexibleNameFormatter()
print(formatter.format_name("Ada"))
print(formatter.format_name("Ada", "Lovelace"))
print(formatter.format_name("Ada", "Lovelace", uppercase=True))

## Übungen zu 4.2

### Aufgabe 1 - Overloading mit flexiblen Parametern

1. Schreibe eine Funktion `build_label()`.
2. Sie soll mindestens die folgenden Aufrufmuster unterstützen:
   ```python
   build_label("Ada")
   build_label("Ada", "Lovelace")
   build_label("Ada", "Lovelace", role="Tutorin")
   build_label(first_name="Ada", last_name="Lovelace")
   build_label("Ada", last_name="Lovelace", city="London")
   build_label(title="Dr.", first_name="Ada", last_name="Lovelace", skills=["Python", "Mathematik"], active=True)
   ```
3. Diese Aufrufmuster sollen hingegen **nicht** funktionieren:
   ```python
   build_label("Ada", "Lovelace", "Tutorin")
   build_label("Ada", "Lovelace", "London")
   build_label(last_name="Lovelace", role="Tutorin")
   ```
3. Formatiere daraus jeweils einen lesbaren String.
4. Werte beliebiege weitere Zusatzinfos aus `**kwargs` aus, z.B. `title`, `city`, `skills`, `active` oder weitere freie Zusatzfelder.
5. Probiere die obigen Aufrufmuster aus und gib die Ergebnisse aus. Teste auch, dass die negativen Muster Fehler werfen.

### Aufgabe 2 - Vererbung, `super()` und abstrakte Basisklassen

1. Schreibe eine abstrakte Basisklasse `Employee` mit dem Attribut `name`.
2. Definiere eine abstrakte Methode `monthly_salary()`.
3. Implementiere `SalariedEmployee` mit festem Monatsgehalt und `HourlyEmployee` mit Stundengehalt und gearbeiteter Stundenzahl, beide mit Aufruf von `super()`.
4. Gib für eine gemischte Liste von Objekten die Monatsgehälter aus.